# 01 · Dual encoder from scratch

Train a **dual encoder** — a text tower, an image tower and one shared projection — with symmetric InfoNCE on plain PyTorch, using the synthetic product corpus. This is the architecture and objective behind `app/embed.py` + `app/projection.py` in production.

The towers are tiny and trained from scratch, not SBERT and CLIP, but everything else is the real thing: L2-normalised embeddings, a bias-free shared projection, in-batch negatives, both retrieval directions. The loss used here is `info_nce` from `app/from_scratch/contrastive.py` — the same function the production trainer calls.

In [ ]:
# --- bootstrap -------------------------------------------------------------
# Working-directory independent, so this notebook runs identically in Jupyter,
# in VS Code and in Google Colab (kernel CWD is /content, and the checkout may
# sit anywhere below it).  Finds the repository, puts services/model-server on
# sys.path, and generates the git-ignored synthetic corpus on first use.
#
# Needs the *repository*, not just this .ipynb: it imports `app` and reads
# `ml/data/generated`.  In a fresh Colab runtime run this once, above or below
# this cell, then re-run:
#
#     !git clone <your-fork-or-url> /content/catalogue-ai
#
# Already cloned elsewhere? Either set CATALOGUE_AI_ROOT=/path/to/catalogue-ai,
# or rely on the scan below.
import os, sys, pathlib

_HINT = pathlib.Path("ml/notebooks/nbsetup.py")


def _is_repo(path):
    return (path / _HINT).is_file()


def _candidates():
    """Ordered places a checkout might be, nearest first."""
    out = []
    env = os.environ.get("CATALOGUE_AI_ROOT")
    if env:
        out.append(pathlib.Path(env).expanduser())
    cwd = pathlib.Path.cwd().resolve()
    out.extend([cwd, *cwd.parents])
    out.extend([pathlib.Path("/content"), pathlib.Path.home()])
    seen, ordered = set(), []
    for p in out:
        try:
            r = p.resolve()
        except OSError:
            continue
        if r not in seen:
            seen.add(r)
            ordered.append(r)
    return ordered


def _find_repo():
    for c in _candidates():
        if _is_repo(c):
            return c
    # shallow scan: a Colab clone can land in any folder below /content,
    # including below a Drive mount
    frontier = [pathlib.Path("/content"), pathlib.Path.home(),
                pathlib.Path.cwd().resolve()]
    for _ in range(4):
        nxt = []
        for d in frontier:
            if _is_repo(d):
                return d
            try:
                kids = [p for p in sorted(d.iterdir())
                        if p.is_dir() and not p.name.startswith(".")
                        and p.name != "__pycache__"]
            except (OSError, PermissionError):
                kids = []
            nxt.extend(kids)
        frontier = nxt
        if not frontier:
            break
    return None


REPO_ROOT = _find_repo()
if REPO_ROOT is None:
    raise SystemExit(
        "catalogue-ai repository not found.\n\n"
        "This notebook needs the repository (it imports `app` and reads\n"
        "ml/data/generated), not just this .ipynb file. In a fresh Colab\n"
        "runtime run this once, then re-run this cell:\n\n"
        "    !git clone <your-fork-or-url> /content/catalogue-ai\n\n"
        "Already cloned elsewhere? Point the notebook at it:\n\n"
        "    import os; os.environ['CATALOGUE_AI_ROOT'] = '/path/to/catalogue-ai'\n\n"
        "searched:\n  " + "\n  ".join(str(c) for c in _candidates()))

sys.path.insert(0, str(REPO_ROOT / "ml" / "notebooks"))
import nbsetup

_summary = nbsetup.bootstrap()
REPO_ROOT = pathlib.Path(_summary["repo_root"])
DATA_DIR = pathlib.Path(_summary["data_dir"])

In [ ]:
import csv, json, hashlib
import numpy as np
import torch
from app.from_scratch.contrastive import info_nce

# CONFIG — scale up for real training
CONFIG = dict(epochs=30, batch=64, d_model=64, dim=64, vocab=4096,
              lr=3e-3, temperature=0.05, seed=42,
              n_items=1024, val_frac=0.25)
torch.manual_seed(CONFIG['seed']); np.random.seed(CONFIG['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| torch', torch.__version__)
print('chance loss for this batch =', round(float(np.log(CONFIG['batch'])), 3))

## Data: hashed tokens + pooled thumbnails

Two details worth flagging, because both were silent bugs:

1. The builtin `hash()` on `str` is **salted per process** (`PYTHONHASHSEED`), so it returns a different vector on every run and would quietly void the `seed` above. `blake2b` is stable across processes, machines and Python versions.
2. Converting to greyscale (`convert('L')`) throws away hue, and hue is exactly what the placeholder photos encode — the model then sits at chance no matter how long it trains. Pooling RGB keeps it.

Images are pooled down to `dim` channels by averaging, so every region of the picture contributes instead of the first 64 pixels.

In [ ]:
from PIL import Image

rows = list(csv.DictReader((DATA_DIR / 'manifest.csv').open()))[:CONFIG['n_items']]
if not rows:
    raise SystemExit('manifest is empty — run ml/data/make_synthetic_dataset.py')

def token_ids(text, vocab=CONFIG['vocab']):
    """Stable hashed token ids — blake2b, never the salted hash()."""
    return [int.from_bytes(hashlib.blake2b(t.encode(), digest_size=8).digest(),
                         'big') % vocab for t in text.lower().split()]

def image_features(path, dim=CONFIG['dim']):
    """16x16 RGB thumbnail average-pooled to exactly `dim` channels."""
    thumb = np.asarray(Image.open(path).convert('RGB')
                        .resize((16, 16), Image.BILINEAR), np.float32).reshape(-1) / 255.0
    edges = np.linspace(0, thumb.size, dim + 1).astype(int)
    return np.array([thumb[a:b].mean() if b > a else 0.0
                      for a, b in zip(edges[:-1], edges[1:])], np.float32)

token_lists = [token_ids(r['description']) for r in rows]
width = max(len(t) for t in token_lists)
TOK = torch.zeros(len(rows), width, dtype=torch.long)
TOK_MASK = torch.zeros(len(rows), width)
for i, toks in enumerate(token_lists):
    TOK[i, :len(toks)] = torch.tensor(toks)
    TOK_MASK[i, :len(toks)] = 1.0
IMG = torch.tensor(np.stack([image_features(DATA_DIR / r['image_path'])
                               for r in rows]))
TOK, TOK_MASK, IMG = TOK.to(device), TOK_MASK.to(device), IMG.to(device)
print('pairs:', tuple(TOK.shape), '| images:', tuple(IMG.shape))

## The dual encoder

Two towers, one shared projection. The projection is **shared**, not separate per modality — that is the whole point: it is the only place the two spaces are forced into agreement, and it is what `app/projection.py` loads at serving time.

In [ ]:
class TinyDualEncoder(torch.nn.Module):
    """text tower + image tower + one shared projection head."""

    def __init__(self, d_model=CONFIG['d_model'], dim=CONFIG['dim'],
                 vocab=CONFIG['vocab']):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab, d_model)
        self.text_tower = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_model), torch.nn.GELU(),
            torch.nn.Linear(d_model, d_model))
        self.image_tower = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_model), torch.nn.GELU(),
            torch.nn.Linear(d_model, d_model))
        # bias-free and shared by both modalities
        self.projection = torch.nn.Linear(d_model, dim, bias=False)

    def _shared(self, tower_out):
        return self.projection(
            torch.nn.functional.normalize(tower_out, dim=-1))

    def encode_text(self, ids, mask):
        m = mask.unsqueeze(-1)
        pooled = (self.embedding(ids) * m).sum(1) / m.sum(1).clamp(min=1.0)
        return self._shared(self.text_tower(pooled))

    def encode_image(self, pixels):
        return self._shared(self.image_tower(pixels))

    def forward(self, ids, mask, pixels):
        return self.encode_text(ids, mask), self.encode_image(pixels)

model = TinyDualEncoder().to(device)
print('parameters:', sum(p.numel() for p in model.parameters()))

## Train with symmetric InfoNCE

The loop keeps **every full batch** — `range(0, n - batch, batch)` silently dropped the last one, and produced *zero* iterations whenever `n <= batch`. A held-out split is carved out first so retrieval is not measured on the rows the towers just memorised.

In [ ]:
n_val = max(CONFIG['batch'], int(len(TOK) * CONFIG['val_frac']))
perm = torch.randperm(len(TOK), device=device)
val_idx, train_idx = perm[:n_val], perm[n_val:]
print(f'train={len(train_idx)} val={len(val_idx)}')

@torch.no_grad()
def retrieval_report(model, idx):
    """Top-1 retrieval both directions over the full NxN similarity matrix.

    scores[i, j] = <text_i, image_j>, so argmax over dim=1 answers
    'given text i, which image is its pair?' (text -> image) and argmax
    over dim=0 answers 'given image j, which text is its pair?' (image -> text).
    """
    t, i = model(TOK[idx], TOK_MASK[idx], IMG[idx])
    scores = t @ i.t()
    gold = torch.arange(scores.shape[0], device=scores.device)
    ranks = (scores > scores[gold, gold].unsqueeze(1)).sum(dim=1) + 1
    return {
        'n': int(scores.shape[0]),
        'chance_top1': round(1.0 / scores.shape[0], 4),
        'acc_text_to_image': round((scores.argmax(1) == gold).float().mean().item(), 4),
        'acc_image_to_text': round((scores.argmax(0) == gold).float().mean().item(), 4),
        'median_rank': int(ranks.float().median().item()),
    }

print('untrained baseline:', json.dumps(retrieval_report(model, val_idx)))

In [ ]:
optimiser = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'])
history = []
for epoch in range(CONFIG['epochs']):
    order = train_idx[torch.randperm(len(train_idx), device=device)]
    for start in range(0, len(order) - CONFIG['batch'] + 1, CONFIG['batch']):
        idx = order[start:start + CONFIG['batch']]
        t_embed, i_embed = model(TOK[idx], TOK_MASK[idx], IMG[idx])
        loss, _ = info_nce(t_embed, i_embed, CONFIG['temperature'])
        optimiser.zero_grad(); loss.backward(); optimiser.step()
        history.append({'epoch': epoch, 'loss': float(loss.detach())})
    epoch_loss = float(np.mean([h['loss'] for h in history if h['epoch'] == epoch]))
    report = retrieval_report(model, val_idx)
    print(json.dumps({'epoch': epoch, 'loss': round(epoch_loss, 4), 'val': report}))

In [ ]:
import matplotlib.pyplot as plt

per_epoch = [float(np.mean([h['loss'] for h in history if h['epoch'] == e]))
             for e in range(CONFIG['epochs'])]
chance = float(np.log(CONFIG['batch']))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(per_epoch, marker='o', label='InfoNCE')
axes[0].axhline(chance, ls='--', c='k', lw=1, label='chance')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss')
axes[0].set_title('shared-space training'); axes[0].legend(fontsize=8)

for name, subset in (('val', val_idx), ('train', train_idx)):
    with torch.no_grad():
        t, i = model(TOK[subset], TOK_MASK[subset], IMG[subset])
        scores = t @ i.t()
        gold = torch.arange(scores.shape[0], device=scores.device)
        ranks = (scores > scores[gold, gold].unsqueeze(1)).sum(1) + 1
    axes[1].plot(np.sort(ranks.float().cpu().numpy()), lw=1,
                 label=f'{name} (n={len(subset)})')
axes[1].axhline(scores.shape[0] / 2, ls='--', c='k', lw=1, label='chance')
axes[1].set_xlabel('query, sorted by rank'); axes[1].set_ylabel('gold rank')
axes[1].set_title('retrieval rank distribution'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## Retrieval report

Held-out first, then train. The gap is the memorisation the objective is optimising; the median rank is the honest headline number, because top-1 stays low while the ranking itself improves sharply. Accuracy is bounded by these towers being 64-dimensional and trained on a synthetic corpus — the real pipeline swaps in SBERT and CLIP, and only the projection and the objective stay the same.

In [ ]:
for label, subset in (('untrained (random init)', None),
                      ('held-out split', val_idx),
                      ('train split (memorisation check)', train_idx)):
    if subset is None:
        torch.manual_seed(0)
        fresh = TinyDualEncoder().to(device)
        report = retrieval_report(fresh, val_idx)
    else:
        report = retrieval_report(model, subset)
    print(f'--- {label} ---')
    print(json.dumps(report, indent=2))